# 02_Data_Quality_and_Preprocessing

**Project:** Taxi Fare Amount Prediction Using NYC Yellow Taxi Big Data

This notebook loads the DataFrame created in `01_Data_Loading_and_Spark_Setup.ipynb`, performs data quality checks, handles missing values, removes duplicates, and filters obvious outliers. The cleaned DataFrame is cached for downstream notebooks.


In [ ]:
# Load the Spark session and DataFrame from previous notebook
%run ./01_Data_Loading_and_Spark_Setup.ipynb

from pyspark.sql import functions as F

# Critical columns for the prediction task
critical_cols = [
    'pickup_datetime',
    'dropoff_datetime',
    'passenger_count',
    'trip_distance',
    'fare_amount'
]

# 1. Missing value analysis
null_counts = df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns])
null_counts.show(truncate=False)

# Drop rows with nulls in critical columns
df_clean = df.dropna(subset=critical_cols)

# 2. Duplicate removal – composite key based on trip timestamps and locations
df_clean = df_clean.dropDuplicates(['pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID', 'trip_distance', 'fare_amount'])

# 3. Basic outlier filtering – remove non‑positive distances / fares
df_clean = df_clean.filter((F.col('trip_distance') > 0) & (F.col('fare_amount') > 0))

# Cache the cleaned DataFrame for faster downstream operations
df_clean.cache()
print(f'Cleaned DataFrame rows: {df_clean.count():,}')
df_clean.printSchema()